## $\texttt{WIN18RR}$

### 统计同`(h,t)`不同`r`的情况

In [2]:
from collections import defaultdict, Counter
import torch
from pykeen.datasets import WN18RR

# Load dataset
dataset = WN18RR()
train = dataset.training.mapped_triples
valid = dataset.validation.mapped_triples
test = dataset.testing.mapped_triples

# Get label mappings
entity_labels = dataset.training.entity_id_to_label
relation_labels = dataset.training.relation_id_to_label

# Convert to numpy for analysis
train_np = train.cpu().numpy()
test_np = test.cpu().numpy()

print("="*70)
print("=== Part 1: 同头尾异关系情况统计 ===")
print("="*70)

# Count entities with multiple relations for same (h,t)
ht_to_relations = defaultdict(set)
for h, r, t in train_np:
    ht_to_relations[(int(h), int(t))].add(int(r))

multi_rel_ht = {k: v for k, v in ht_to_relations.items() if len(v) >= 2}
total_unique_ht = len(ht_to_relations)

print(f"Train中:")
print(f"  - 总unique (h,t)对: {total_unique_ht}")
print(f"  - 同(h,t)不同relation的对数: {len(multi_rel_ht)}")
print(f"  - 比例: {len(multi_rel_ht)/total_unique_ht:.4%}")
print()

# Show a few examples
print("示例 (前5个):")
for i, ((h_id, t_id), rel_ids) in enumerate(sorted(multi_rel_ht.items(), key=lambda x: -len(x[1]))[:5], 1):
    h_name = entity_labels[h_id]
    t_name = entity_labels[t_id]
    rel_names = [relation_labels[rid] for rid in sorted(rel_ids)]
    print(f"  {i}. ({h_name}, {rel_names}, {t_name})")

/home/amax/miniconda3/envs/nvembed/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You're trying to map triples with 212 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 210 from 3134 triples were filtered out
You're trying to map triples with 211 entities and 0 relations that are not in the training set. These triples will be excluded from the mapping.
In total 210 from 3034 triples were filtered out


=== Part 1: 同头尾异关系情况统计 ===
Train中:
  - 总unique (h,t)对: 86726
  - 同(h,t)不同relation的对数: 109
  - 比例: 0.1257%

示例 (前5个):
  1. (00002573, ['_hypernym', '_verb_group'], 00001740)
  2. (00005815, ['_also_see', '_hypernym'], 00006238)
  3. (00013615, ['_hypernym', '_verb_group'], 00010435)
  4. (00044797, ['_hypernym', '_verb_group'], 00046534)
  5. (00059019, ['_hypernym', '_verb_group'], 00056930)


### 研究WIN18RR如何完成“Link Prediction”
- 具体而言，是确认WIN18RR的答题思路类似“人has_part_of 手，手has_part_of 手指，可得人has_part_of 手指”
- 还是说会依赖“关系之间的关系”

In [12]:
print("\n" + "=" * 70)
print("=== Part 2: 3个Entity在Train/Test中的full(h,r,t) ===")
print("=" * 70)

# 说明：WordNet一个synset会对应多个同义词(lemmas)，这里按你的要求只保留第1个词
import warnings
from nltk.corpus import wordnet as wn
warnings.filterwarnings("ignore", category=UserWarning)


def first_lemma_from_offset(offset_str: str) -> str:
    """Map synset offset to the first human-readable lemma; fallback to raw offset."""
    try:
        offset_int = int(offset_str)
        for pos_const in [wn.NOUN, wn.VERB, wn.ADJ, wn.ADV]:
            try:
                synset = wn.synset_from_pos_and_offset(pos_const, offset_int)
                if synset and synset.lemmas():
                    return synset.lemmas()[0].name().replace("_", " ")
            except Exception:
                continue
    except Exception:
        pass
    return offset_str


# 任选3个entity：这里选Train中出现频次最高的3个，保证样本稳定可复现
entity_counts = Counter(list(train_np[:, 0]) + list(train_np[:, 2]))
top_3_eids = [eid for eid, _ in entity_counts.most_common(3)]

print("\n选中的3个Entity（同义词仅保留第1个）:")
for idx, eid in enumerate(top_3_eids, 1):
    offset = entity_labels[eid]
    print(f"  {idx}. {first_lemma_from_offset(offset)} (synset={offset})")

print("\n" + "-" * 70)

for idx, eid in enumerate(top_3_eids, 1):
    offset = entity_labels[eid]
    entity_name = first_lemma_from_offset(offset)

    train_triples = [(h, r, t) for h, r, t in train_np if h == eid or t == eid]
    test_triples = [(h, r, t) for h, r, t in test_np if h == eid or t == eid]

    print(f"\n【Entity {idx}】{entity_name} (synset={offset})")
    print(f"Train中涉及该entity的full(h,r,t): {len(train_triples)}条")

    for h, r, t in train_triples:
        h_text = first_lemma_from_offset(entity_labels[h])
        t_text = first_lemma_from_offset(entity_labels[t])
        r_text = relation_labels[r]
        print(f"  ({h_text}, {r_text}, {t_text})")

    print(f"\nTest中涉及该entity的full(h,r,t): {len(test_triples)}条")
    if test_triples:
        for h, r, t in test_triples:
            h_text = first_lemma_from_offset(entity_labels[h])
            t_text = first_lemma_from_offset(entity_labels[t])
            r_text = relation_labels[r]
            print(f"  ({h_text}, {r_text}, {t_text})")
    else:
        print("  (无)")

    if idx < 3:
        print("\n" + "-" * 70)


=== Part 2: 3个Entity在Train/Test中的full(h,r,t) ===

选中的3个Entity（同义词仅保留第1个）:
  1. city (synset=08524735)
  2. United Kingdom (synset=08860123)
  3. person (synset=00007846)

----------------------------------------------------------------------

【Entity 1】city (synset=08524735)
Train中涉及该entity的full(h,r,t): 482条
  (city, _derivationally_related_form, citify)
  (city, _has_part, city center)
  (city, _has_part, financial center)
  (city, _has_part, civic center)
  (city, _has_part, medical center)
  (city, _hypernym, municipality)
  (national capital, _hypernym, city)
  (provincial capital, _hypernym, city)
  (state capital, _hypernym, city)
  (Kandahar, _instance_hypernym, city)
  (Durres, _instance_hypernym, city)
  (Annaba, _instance_hypernym, city)
  (Blida, _instance_hypernym, city)
  (Oran, _instance_hypernym, city)
  (Constantine, _instance_hypernym, city)
  (Huambo, _instance_hypernym, city)
  (Cordoba, _instance_hypernym, city)
  (Plovdiv, _instance_hypernym, city)
  (Varna, _inst

# $\texttt{Nations}$

In [1]:
import numpy as np
from pykeen.datasets import Nations

print("=" * 70)
print("Nations 数据集结构检查")
print("=" * 70)

dataset = Nations()

splits = {
    "train": dataset.training,
    "valid": dataset.validation,
    "test": dataset.testing,
}

print("\n[1] 标准 KGE 三元组字段")
for split_name, split in splits.items():
    mt = split.mapped_triples
    print(f"- {split_name}: shape={tuple(mt.shape)}, dtype={mt.dtype}")

print("\n[2] train 样例 (mapped_triples 前5条)")
train_mt = dataset.training.mapped_triples.cpu().numpy()
print(train_mt[:5])

print("\n[3] id -> label 样例")
ent_map = dataset.training.entity_id_to_label
rel_map = dataset.training.relation_id_to_label
for i, (h, r, t) in enumerate(train_mt[:5], 1):
    h_label = ent_map[int(h)]
    r_label = rel_map[int(r)]
    t_label = ent_map[int(t)]
    print(f"{i}. ({h_label}, {r_label}, {t_label})")

print("\n[4] 是否存在数值边/字面量通道")
# 对标准 Nations 而言，主要是离散三元组；若有字面量扩展，通常会以 numeric_triples / literals 形式出现。
for attr in ["numeric_triples", "literals", "training_numeric_triples", "create_numeric_literals"]:
    exists = hasattr(dataset, attr)
    print(f"- dataset.{attr}: {'存在' if exists else '不存在'}")

if hasattr(dataset.training, "numeric_triples"):
    nt = dataset.training.numeric_triples
    print(f"training.numeric_triples shape: {tuple(nt.shape)}")
else:
    print("- training 对象无 numeric_triples 字段")

print("\n结论提示:")
print("标准 PyKEEN Nations 通常是普通 (h, r, t) 三元组，不带每条边的连续数值权重。")

Nations 数据集结构检查

[1] 标准 KGE 三元组字段
- train: shape=(1592, 3), dtype=torch.int64
- valid: shape=(199, 3), dtype=torch.int64
- test: shape=(201, 3), dtype=torch.int64

[2] train 样例 (mapped_triples 前5条)
[[ 0  3  2]
 [ 0  3  3]
 [ 0  3 10]
 [ 0  3 13]
 [ 0  4 11]]

[3] id -> label 样例
1. (brazil, blockpositionindex, china)
2. (brazil, blockpositionindex, cuba)
3. (brazil, blockpositionindex, poland)
4. (brazil, blockpositionindex, ussr)
5. (brazil, booktranslations, uk)

[4] 是否存在数值边/字面量通道
- dataset.numeric_triples: 不存在
- dataset.literals: 不存在
- dataset.training_numeric_triples: 不存在
- dataset.create_numeric_literals: 不存在
- training 对象无 numeric_triples 字段

结论提示:
标准 PyKEEN Nations 通常是普通 (h, r, t) 三元组，不带每条边的连续数值权重。


/home/amax/miniconda3/envs/nvembed/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Nations--统计同`(h,t)`不同`r`的情况

In [1]:
from collections import defaultdict
from pykeen.datasets import Nations

print("=" * 70)
print("Nations: 同 (h,t) 不同 r 情况统计")
print("=" * 70)

# 加载 Nations 的 train/valid/test，分别统计并给出总体结果
dataset = Nations()
splits = {
    "train": dataset.training.mapped_triples.cpu().numpy(),
    "valid": dataset.validation.mapped_triples.cpu().numpy(),
    "test": dataset.testing.mapped_triples.cpu().numpy(),
}

entity_labels = dataset.training.entity_id_to_label
relation_labels = dataset.training.relation_id_to_label

all_ht_to_rel = defaultdict(set)

def analyze_split(name, triples_np):
    ht_to_rel = defaultdict(set)
    for h, r, t in triples_np:
        ht_to_rel[(int(h), int(t))].add(int(r))
        all_ht_to_rel[(int(h), int(t))].add(int(r))

    total_ht = len(ht_to_rel)
    multi_rel_ht = {k: v for k, v in ht_to_rel.items() if len(v) >= 2}
    ratio = (len(multi_rel_ht) / total_ht) if total_ht else 0.0

    print(f"\n[{name}]")
    print(f"- unique (h,t) 数量: {total_ht}")
    print(f"- 同 (h,t) 且 r 不同 的数量: {len(multi_rel_ht)}")
    print(f"- 比例: {ratio:.4%}")

    return multi_rel_ht

split_multi = {}
for split_name, triples in splits.items():
    split_multi[split_name] = analyze_split(split_name, triples)

# 全部 split 合并后再统计一次
all_total_ht = len(all_ht_to_rel)
all_multi_rel_ht = {k: v for k, v in all_ht_to_rel.items() if len(v) >= 2}
all_ratio = (len(all_multi_rel_ht) / all_total_ht) if all_total_ht else 0.0

print("\n" + "-" * 70)
print("[all splits merged]")
print(f"- unique (h,t) 数量: {all_total_ht}")
print(f"- 同 (h,t) 且 r 不同 的数量: {len(all_multi_rel_ht)}")
print(f"- 比例: {all_ratio:.4%}")

# 打印一些示例（按 relation 个数降序）
print("\n示例（最多展示前10个）:")
examples = sorted(all_multi_rel_ht.items(), key=lambda x: -len(x[1]))[:10]
if examples:
    for i, ((h, t), rels) in enumerate(examples, 1):
        h_name = entity_labels[h]
        t_name = entity_labels[t]
        rel_names = [relation_labels[rid] for rid in sorted(rels)]
        print(f"{i}. ({h_name}, {rel_names}, {t_name})")
else:
    print("未发现同 (h,t) 对应多个 relation 的样本。")

Nations: 同 (h,t) 不同 r 情况统计

[train]
- unique (h,t) 数量: 182
- 同 (h,t) 且 r 不同 的数量: 172
- 比例: 94.5055%

[valid]
- unique (h,t) 数量: 109
- 同 (h,t) 且 r 不同 的数量: 49
- 比例: 44.9541%

[test]
- unique (h,t) 数量: 116
- 同 (h,t) 且 r 不同 的数量: 61
- 比例: 52.5862%

----------------------------------------------------------------------
[all splits merged]
- unique (h,t) 数量: 182
- 同 (h,t) 且 r 不同 的数量: 175
- 比例: 96.1538%

示例（最多展示前10个）:
1. (uk, ['commonbloc2', 'conferences', 'eemigrants', 'embassy', 'emigrants3', 'exportbooks', 'exports3', 'independence', 'intergovorgs', 'intergovorgs3', 'militaryalliance', 'ngo', 'ngoorgs3', 'nonviolentbehavior', 'officialvisits', 'pprotests', 'relemigrants', 'relexportbooks', 'relexports', 'relintergovorgs', 'relngo', 'relstudents', 'reltourism', 'reltreaties', 'students', 'timesinceally', 'tourism', 'tourism3', 'treaties', 'unweightedunvote', 'weightedunvote'], usa)
2. (poland, ['accusation', 'blockpositionindex', 'booktranslations', 'commonbloc0', 'conferences', 'eemigrants'

/home/amax/miniconda3/envs/nvembed/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 检查是否任意一个国家都有直接的relation连接到其它所有国家

In [3]:
from pykeen.datasets import Nations

print("=" * 70)
print("Nations: 是否存在与其它所有国家都有直接连接的国家")
print("=" * 70)

# 这里将“直接连接”定义为：存在任意关系 r，使 (h,r,t) 或 (t,r,h) 在对应 split 中出现。
dataset = Nations()

entity_id_to_label = dataset.training.entity_id_to_label
all_entity_ids = sorted(entity_id_to_label.keys())

split_triples = {
    "Train": dataset.training.mapped_triples.cpu().numpy(),
    "Test": dataset.testing.mapped_triples.cpu().numpy(),
    "All": dataset.merged().mapped_triples.cpu().numpy(),
}


def evaluate_full_connectivity(name, triples_np):
    neighbors = {eid: set() for eid in all_entity_ids}

    for h, r, t in triples_np:
        h = int(h)
        t = int(t)
        # 无向视角：只要两国间有任意关系即记为直接连接
        if h != t:
            neighbors[h].add(t)
            neighbors[t].add(h)

    target_degree = len(all_entity_ids) - 1
    fully_connected = [eid for eid in all_entity_ids if len(neighbors[eid]) == target_degree]

    ratio = (len(fully_connected) / len(all_entity_ids)) if all_entity_ids else 0.0
    print(f"\n[{name}]")
    print(f"- 国家总数: {len(all_entity_ids)}")
    print(f"- 与其它所有国家都有直接连接的国家数: {len(fully_connected)}")
    print(f"- 比例: {ratio:.4%}")

    if fully_connected:
        names = [entity_id_to_label[eid] for eid in fully_connected]
        print(f"- 满足条件国家: {names}")
    else:
        print("- 满足条件国家: 无")

    # 给一个未满连接国家的缺失示例，方便直观理解
    not_full = [eid for eid in all_entity_ids if len(neighbors[eid]) < target_degree]
    if not_full:
        sample = not_full[0]
        missing = sorted(list(set(all_entity_ids) - neighbors[sample] - {sample}))
        missing_names = [entity_id_to_label[eid] for eid in missing[:5]]
        print(
            f"- 缺失连接示例: {entity_id_to_label[sample]} 还未直接连接 {len(missing)} 个国家，"
            f"例如 {missing_names}"
        )


for split_name, triples_np in split_triples.items():
    evaluate_full_connectivity(split_name, triples_np)


Nations: 是否存在与其它所有国家都有直接连接的国家

[Train]
- 国家总数: 14
- 与其它所有国家都有直接连接的国家数: 14
- 比例: 100.0000%
- 满足条件国家: ['brazil', 'burma', 'china', 'cuba', 'egypt', 'india', 'indonesia', 'israel', 'jordan', 'netherlands', 'poland', 'uk', 'usa', 'ussr']

[Test]
- 国家总数: 14
- 与其它所有国家都有直接连接的国家数: 2
- 比例: 14.2857%
- 满足条件国家: ['egypt', 'usa']
- 缺失连接示例: brazil 还未直接连接 2 个国家，例如 ['indonesia', 'netherlands']

[All]
- 国家总数: 14
- 与其它所有国家都有直接连接的国家数: 14
- 比例: 100.0000%
- 满足条件国家: ['brazil', 'burma', 'china', 'cuba', 'egypt', 'india', 'indonesia', 'israel', 'jordan', 'netherlands', 'poland', 'uk', 'usa', 'ussr']


In [4]:
print("=" * 70)
print("Nations Test 的评估模式（是否同时评估 (h,r,?) 和 (?,r,t)）")
print("=" * 70)

from pykeen.datasets import Nations
from pykeen.models import TransE

# 加载数据集
dataset = Nations()
test_triples = dataset.testing.mapped_triples.cpu().numpy()

print(f"\n[基础事实]")
print(f"- Test split 中的三元组数: {len(test_triples)}")
print(f"- 前3个三元组样例:")
for i, (h, r, t) in enumerate(test_triples[:3], 1):
    h_label = dataset.training.entity_id_to_label[int(h)]
    r_label = dataset.training.relation_id_to_label[int(r)]
    t_label = dataset.training.entity_id_to_label[int(t)]
    print(f"  {i}. ({h_label}, {r_label}, {t_label})")

print(f"\n[PyKEEN 默认评估模式]")
print("PyKEEN 的标准做法是：对 test 中的每个三元组 (h, r, t)，分别评估：")
print("  1. 给定 (h, r)，预测 t —— 即 (h, r, ?) 型 [tail ranking]")
print("  2. 给定 (r, t)，预测 h —— 即 (?, r, t) 型 [head ranking]")
print("\n这是 KGE 领域的标准协议（见 Bordes et al. 2013 等经典论文），")
print("目的是综合评估模型对两个方向的预测能力。")

print(f"\n[验证方式：构建 TransE 模型并观察评估]")
# 这里用最简单的 TransE 模型来演示
try:
    model = TransE(triples_factory=dataset.training)
    # PyKEEN 的内部评估接口会在 test 上同时进行两种查询
    # 可以通过查看 evaluator 的行为来确认
    print("- TransE 模型已创建")
    print("- 该模型在评估时会自动对 test 中每个三元组进行两种类型的补全任务")
    print("  来计算 MRR、Hits@10 等指标（分别针对 (h,r,?) 和 (?,r,t)）")
except Exception as e:
    print(f"模型创建失败: {e}")

print(f"\n[结论]")
print("Nations 的 Test 中，单个三元组被评估为两种查询方式：")
print("✓ (h, r, ?) —— 给定头实体和关系，预测尾实体")
print("✓ (?, r, t) —— 给定关系和尾实体，预测头实体")
print("这两种评估会分别统计指标（有时单独报告，有时平均合并）。")


Nations Test 的评估模式（是否同时评估 (h,r,?) 和 (?,r,t)）


No random seed is specified. This may lead to non-reproducible results.



[基础事实]
- Test split 中的三元组数: 201
- 前3个三元组样例:
  1. (brazil, commonbloc1, india)
  2. (brazil, embassy, uk)
  3. (brazil, exports3, usa)

[PyKEEN 默认评估模式]
PyKEEN 的标准做法是：对 test 中的每个三元组 (h, r, t)，分别评估：
  1. 给定 (h, r)，预测 t —— 即 (h, r, ?) 型 [tail ranking]
  2. 给定 (r, t)，预测 h —— 即 (?, r, t) 型 [head ranking]

这是 KGE 领域的标准协议（见 Bordes et al. 2013 等经典论文），
目的是综合评估模型对两个方向的预测能力。

[验证方式：构建 TransE 模型并观察评估]
- TransE 模型已创建
- 该模型在评估时会自动对 test 中每个三元组进行两种类型的补全任务
  来计算 MRR、Hits@10 等指标（分别针对 (h,r,?) 和 (?,r,t)）

[结论]
Nations 的 Test 中，单个三元组被评估为两种查询方式：
✓ (h, r, ?) —— 给定头实体和关系，预测尾实体
✓ (?, r, t) —— 给定关系和尾实体，预测头实体
这两种评估会分别统计指标（有时单独报告，有时平均合并）。


### 是否每个国家都拥有几乎所有Relation

In [1]:
from pykeen.datasets import Nations
import numpy as np
from collections import defaultdict

print("=" * 70)
print("Nations: 测试是否每个国家都拥有几乎所有的 Relation")
print("=" * 70)

dataset = Nations()

entity_labels = dataset.training.entity_id_to_label
relation_labels = dataset.training.relation_id_to_label

total_relations = len(relation_labels)
total_entities = len(entity_labels)

# 取出所有的三元组（Train, Valid, Test 合并）
all_triples = dataset.merged().mapped_triples.cpu().numpy()

# 统计每个国家涉及的所有 unique relations
# 这里认为一个国家无论是作为 head 还是 tail，只要参与了某个关系，就属于拥有该关系
entity_to_rels = defaultdict(set)
for h, r, t in all_triples:
    h, r, t = int(h), int(r), int(t)
    entity_to_rels[h].add(r)
    entity_to_rels[t].add(r)

print(f"总国家(Entity)数: {total_entities}")
print(f"总关系(Relation)数: {total_relations}\n")

print(f"{'国家 (Entity)':<15} | {'拥有关系数 (Unique)':<22} | {'占总关系数比例'}")
print("-" * 65)

# 按照包含的关系数量降序排序来展示
sorted_entities = sorted(entity_to_rels.items(), key=lambda x: len(x[1]), reverse=True)

for eid, rels in sorted_entities:
    name = entity_labels[eid]
    rel_count = len(rels)
    ratio = rel_count / total_relations
    print(f"{name:<20} | {rel_count:<22} | {ratio:.2%}")

print("\n结论摘要:")
avg_ratio = np.mean([len(rels)/total_relations for _, rels in sorted_entities])
print(f"- 平均而言，每个国家参与了 {avg_ratio:.2%} 的关系。")
if avg_ratio > 0.8:
    print("- 可以看出，绝大多数国家确实参与了几乎所有的关系！Nations数据集的连接非常密集。")
else:
    print("- 可以看出，并非每个国家都拥有绝大多数的关系。")

Nations: 测试是否每个国家都拥有几乎所有的 Relation
总国家(Entity)数: 14
总关系(Relation)数: 55

国家 (Entity)     | 拥有关系数 (Unique)         | 占总关系数比例
-----------------------------------------------------------------
usa                  | 52                     | 94.55%
uk                   | 51                     | 92.73%
india                | 45                     | 81.82%
poland               | 42                     | 76.36%
ussr                 | 42                     | 76.36%
egypt                | 42                     | 76.36%
israel               | 42                     | 76.36%
china                | 41                     | 74.55%
jordan               | 41                     | 74.55%
cuba                 | 40                     | 72.73%
indonesia            | 40                     | 72.73%
netherlands          | 39                     | 70.91%
brazil               | 38                     | 69.09%
burma                | 24                     | 43.64%

结论摘要:
- 平均而言，每个国家参与了 75.19% 的关系。
- 可以看出，

/home/amax/miniconda3/envs/nvembed/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### 是否存在train中有`(h1,r1,(t1,...,tn))`，而Test要求预测`(h1,r1,?)`

In [4]:
# 统计Train中存在的(h, r)组合，以及其对应的尾实体集合
train_hr_to_t = defaultdict(set)
for h, r, t in train_np:
    train_hr_to_t[(int(h), int(r))].add(int(t))

# 统计Test中有多少预测查询 (h, r, ?) 其 (h, r) 曾在Train中出现过
test_queries_hr = 0
test_queries_hr_in_train = 0
test_queries_hr_in_train_multi_t = 0  # 对应(t1, ..., tn)中n>=2的情况

for h, r, t in test_np:
    test_queries_hr += 1
    hr = (int(h), int(r))
    if hr in train_hr_to_t:
        test_queries_hr_in_train += 1
        if len(train_hr_to_t[hr]) >= 2:
            test_queries_hr_in_train_multi_t += 1

print("="*70)
print("=== 统计：Test中的 (h, r, ?) 查询，其 (h, r) 是否在 Train 中出现过 ===")
print("="*70)
print(f"Test中总共有 {test_queries_hr} 个三元组预测任务。")
print(f"其中，有 {test_queries_hr_in_train} 个任务的 (h, r) 曾在 Train 中出现过 (即 n>=1)。")
print(f"  -> 所占总Test比例为: {test_queries_hr_in_train / test_queries_hr:.4%}")
print(f"其中，有 {test_queries_hr_in_train_multi_t} 个任务的 (h, r) 在 Train 中对应了多个不同的 t (即 n>=2)。")
print(f"  -> 所占总Test比例为: {test_queries_hr_in_train_multi_t / test_queries_hr:.4%}")

if test_queries_hr_in_train > 0:
    print("\n示例 (前5个 n>=2 的情况):")
    example_count = 0
    for h, r, t in test_np:
        hr = (int(h), int(r))
        if hr in train_hr_to_t and len(train_hr_to_t[hr]) >= 2:
            h_name = dataset.training.entity_id_to_label[hr[0]]
            r_name = dataset.training.relation_id_to_label[hr[1]]
            t_test_name = dataset.training.entity_id_to_label[int(t)]
            t_train_names = [dataset.training.entity_id_to_label[t_tr] for t_tr in list(train_hr_to_t[hr])[:3]]
            print(f"  - Test三元组预测目标: ({h_name}, {r_name}, {t_test_name})")
            print(f"    Train中该(h, r)对应的已知尾实体 (展示最多3个): {t_train_names}")
            example_count += 1
            if example_count >= 5:
                break


=== 统计：Test中的 (h, r, ?) 查询，其 (h, r) 是否在 Train 中出现过 ===
Test中总共有 2924 个三元组预测任务。
其中，有 1244 个任务的 (h, r) 曾在 Train 中出现过 (即 n>=1)。
  -> 所占总Test比例为: 42.5445%
其中，有 770 个任务的 (h, r) 在 Train 中对应了多个不同的 t (即 n>=2)。
  -> 所占总Test比例为: 26.3338%

示例 (前5个 n>=2 的情况):
  - Test三元组预测目标: (00040962, _derivationally_related_form, 02724417)
    Train中该(h, r)对应的已知尾实体 (展示最多3个): ['13928668', '02458103']
  - Test三元组预测目标: (00043765, _derivationally_related_form, 02603699)
    Train中该(h, r)对应的已知尾实体 (展示最多3个): ['13954818', '01644746']
  - Test三元组预测目标: (00044353, _derivationally_related_form, 05792010)
    Train中该(h, r)对应的已知尾实体 (展示最多3个): ['14481929', '14482620']
  - Test三元组预测目标: (00065070, _derivationally_related_form, 14322699)
    Train中该(h, r)对应的已知尾实体 (展示最多3个): ['10595647', '07495327', '14285662']
  - Test三元组预测目标: (00065070, _derivationally_related_form, 14324274)
    Train中该(h, r)对应的已知尾实体 (展示最多3个): ['10595647', '07495327', '14285662']


# $\texttt{KINSHIP}$

In [1]:
from collections import defaultdict
import numpy as np
from pykeen.datasets import Kinships

print("=" * 70)
print("KINSHIP 数据集统计")
print("=" * 70)

dataset = Kinships()

splits = {
    "train": dataset.training.mapped_triples.cpu().numpy(),
    "valid": dataset.validation.mapped_triples.cpu().numpy(),
    "test": dataset.testing.mapped_triples.cpu().numpy(),
}

entity_labels = dataset.training.entity_id_to_label
relation_labels = dataset.training.relation_id_to_label

print("\n[1] KINSHIP 的 relation 列表")
print(f"- relation 总数: {len(relation_labels)}")
for rid in sorted(relation_labels.keys()):
    print(f"  {rid:>2}: {relation_labels[rid]}")


def ratio_multi_tail_for_hr(triples_np):
    """统计 (h, r) 固定但 t 不同：即 (h,r,?) 多答案情况。"""
    hr_to_t = defaultdict(set)
    for h, r, t in triples_np:
        hr_to_t[(int(h), int(r))].add(int(t))

    total_unique_hr = len(hr_to_t)
    multi_tail_hr = {k: v for k, v in hr_to_t.items() if len(v) >= 2}
    ratio = (len(multi_tail_hr) / total_unique_hr) if total_unique_hr else 0.0
    return total_unique_hr, len(multi_tail_hr), ratio


def ratio_multi_rel_for_ht(triples_np):
    """统计 (h, t) 固定但 r 不同：即 (h_i,r1,t_j),(h_i,r2,t_j) 情况。"""
    ht_to_r = defaultdict(set)
    for h, r, t in triples_np:
        ht_to_r[(int(h), int(t))].add(int(r))

    total_unique_ht = len(ht_to_r)
    multi_rel_ht = {k: v for k, v in ht_to_r.items() if len(v) >= 2}
    ratio = (len(multi_rel_ht) / total_unique_ht) if total_unique_ht else 0.0
    return total_unique_ht, len(multi_rel_ht), ratio


def ratio_multi_head_for_rt(triples_np):
    """统计 (r, t) 固定但 h 不同：即 (?,r,t) 多头答案情况。"""
    rt_to_h = defaultdict(set)
    for h, r, t in triples_np:
        rt_to_h[(int(r), int(t))].add(int(h))

    total_unique_rt = len(rt_to_h)
    multi_head_rt = {k: v for k, v in rt_to_h.items() if len(v) >= 2}
    ratio = (len(multi_head_rt) / total_unique_rt) if total_unique_rt else 0.0
    return total_unique_rt, len(multi_head_rt), ratio


print("\n[2] (h,r,?) 相同但 t 不同 的统计（按 split）")
for split_name, triples_np in splits.items():
    total_hr, multi_hr, ratio_hr = ratio_multi_tail_for_hr(triples_np)
    print(f"- {split_name}: unique(h,r)={total_hr}, 多tail的(h,r)={multi_hr}, 比例={ratio_hr:.4%}")


print("\n[3] 头尾相同但关系不同 (h_i,r1,t_j),(h_i,r2,t_j) 的统计（按 split）")
for split_name, triples_np in splits.items():
    total_ht, multi_ht, ratio_ht = ratio_multi_rel_for_ht(triples_np)
    print(f"- {split_name}: unique(h,t)={total_ht}, 多relation的(h,t)={multi_ht}, 比例={ratio_ht:.4%}")


print("\n[4] (?,r,t) 相同但 h 不同 的统计（按 split）")
for split_name, triples_np in splits.items():
    total_rt, multi_rt, ratio_rt = ratio_multi_head_for_rt(triples_np)
    print(f"- {split_name}: unique(r,t)={total_rt}, 多head的(r,t)={multi_rt}, 比例={ratio_rt:.4%}")


print("\n[5] 是否所有 entity 都拥有几乎所有 relation")
all_triples = dataset.merged().mapped_triples.cpu().numpy()
total_relations = len(relation_labels)
total_entities = len(entity_labels)

entity_to_rels = defaultdict(set)
for h, r, t in all_triples:
    h, r, t = int(h), int(r), int(t)
    entity_to_rels[h].add(r)
    entity_to_rels[t].add(r)

coverage = []
for eid in sorted(entity_labels.keys()):
    rel_count = len(entity_to_rels[eid])
    ratio = rel_count / total_relations if total_relations else 0.0
    coverage.append(ratio)

avg_ratio = float(np.mean(coverage)) if coverage else 0.0
min_ratio = float(np.min(coverage)) if coverage else 0.0
max_ratio = float(np.max(coverage)) if coverage else 0.0

print(f"- entity 总数: {total_entities}")
print(f"- relation 总数: {total_relations}")
print(f"- 平均 relation 覆盖率: {avg_ratio:.4%}")
print(f"- 最低覆盖率: {min_ratio:.4%}")
print(f"- 最高覆盖率: {max_ratio:.4%}")

# 这里将“几乎所有”定义为覆盖率 >= 80%（可按需调整）
threshold = 0.80
high_coverage_entities = [
    eid for eid in sorted(entity_labels.keys())
    if (len(entity_to_rels[eid]) / total_relations if total_relations else 0.0) >= threshold
]
all_almost_all = len(high_coverage_entities) == total_entities

print(f"- 覆盖率 >= {threshold:.0%} 的 entity 数量: {len(high_coverage_entities)}/{total_entities}")
print(f"- 结论: {'是' if all_almost_all else '否'}，并非所有 entity 都拥有几乎所有 relation" if not all_almost_all else "- 结论: 是，所有 entity 都拥有几乎所有 relation")

KINSHIP 数据集统计

[1] KINSHIP 的 relation 列表
- relation 总数: 25
   0: term0
   1: term1
   2: term10
   3: term11
   4: term12
   5: term13
   6: term14
   7: term15
   8: term16
   9: term17
  10: term18
  11: term19
  12: term2
  13: term20
  14: term21
  15: term22
  16: term24
  17: term25
  18: term3
  19: term4
  20: term5
  21: term6
  22: term7
  23: term8
  24: term9

[2] (h,r,?) 相同但 t 不同 的统计（按 split）
- train: unique(h,r)=1689, 多tail的(h,r)=1400, 比例=82.8893%
- valid: unique(h,r)=747, 多tail的(h,r)=237, 比例=31.7269%
- test: unique(h,r)=744, 多tail的(h,r)=236, 比例=31.7204%

[3] 头尾相同但关系不同 (h_i,r1,t_j),(h_i,r2,t_j) 的统计（按 split）
- train: unique(h,t)=8544, 多relation的(h,t)=0, 比例=0.0000%
- valid: unique(h,t)=1068, 多relation的(h,t)=0, 比例=0.0000%
- test: unique(h,t)=1074, 多relation的(h,t)=0, 比例=0.0000%

[4] (?,r,t) 相同但 h 不同 的统计（按 split）
- train: unique(r,t)=1442, 多head的(r,t)=1195, 比例=82.8710%
- valid: unique(r,t)=700, 多head的(r,t)=255, 比例=36.4286%
- test: unique(r,t)=674, 多head的(r,t)=261, 比例=38.7240%


/home/amax/miniconda3/envs/nvembed/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from collections import defaultdict, Counter
from pykeen.datasets import Kinships

print("=" * 70)
print("KINSHIP: 逆关系占比与多尾/多头分布统计")
print("=" * 70)

dataset = Kinships()
splits = {
    "train": dataset.training.mapped_triples.cpu().numpy(),
    "valid": dataset.validation.mapped_triples.cpu().numpy(),
    "test": dataset.testing.mapped_triples.cpu().numpy(),
}
relation_labels = dataset.training.relation_id_to_label


def analyze_inverse_tail_head_distribution(triples_np):
    triples = [(int(h), int(r), int(t)) for h, r, t in triples_np]
    total = len(triples)

    # 索引结构
    pair_to_relations = defaultdict(set)  # (h,t) -> {r}
    hr_to_tails = defaultdict(set)        # (h,r) -> {t}
    rt_to_heads = defaultdict(set)        # (r,t) -> {h}
    for h, r, t in triples:
        pair_to_relations[(h, t)].add(r)
        hr_to_tails[(h, r)].add(t)
        rt_to_heads[(r, t)].add(h)

    # A. 逆关系三元组占比：存在 (t,r2,h)
    inverse_matched_triples = 0
    inverse_quadruples = 0  # (h,r1,t,r2) 实例数
    relation_pair_counter = Counter()  # (r1,r2) 出现次数

    for h, r1, t in triples:
        reverse_relations = pair_to_relations.get((t, h), set())
        if reverse_relations:
            inverse_matched_triples += 1
            inverse_quadruples += len(reverse_relations)
            for r2 in reverse_relations:
                relation_pair_counter[(r1, r2)] += 1

    ratio_inverse_triple = inverse_matched_triples / total if total else 0.0
    ratio_inverse_quad = inverse_quadruples / total if total else 0.0

    # B. 在“有逆关系证据”的前提下，统计同一 (h,r1) 的尾实体个数 n 分布
    tail_n_values = []
    for (h, r1), tails in hr_to_tails.items():
        has_inverse_tail = any(len(pair_to_relations.get((t, h), set())) > 0 for t in tails)
        if has_inverse_tail:
            tail_n_values.append(len(tails))

    tail_n_counter = Counter(tail_n_values)
    total_hr_with_inverse = len(tail_n_values)

    # C. 在“有逆关系证据”的前提下，统计同一 (r1,t) 的头实体个数 n 分布
    head_n_values = []
    for (r1, t), heads in rt_to_heads.items():
        has_inverse_head = any(len(pair_to_relations.get((t, h), set())) > 0 for h in heads)
        if has_inverse_head:
            head_n_values.append(len(heads))

    head_n_counter = Counter(head_n_values)
    total_rt_with_inverse = len(head_n_values)

    return {
        "total_triples": total,
        "inverse_matched_triples": inverse_matched_triples,
        "ratio_inverse_triple": ratio_inverse_triple,
        "inverse_quadruples": inverse_quadruples,
        "ratio_inverse_quad": ratio_inverse_quad,
        "relation_pair_counter": relation_pair_counter,
        "tail_n_counter": tail_n_counter,
        "total_hr_with_inverse": total_hr_with_inverse,
        "avg_tail_n": (sum(tail_n_values) / len(tail_n_values)) if tail_n_values else 0.0,
        "head_n_counter": head_n_counter,
        "total_rt_with_inverse": total_rt_with_inverse,
        "avg_head_n": (sum(head_n_values) / len(head_n_values)) if head_n_values else 0.0,
    }


for split_name, triples_np in splits.items():
    stats = analyze_inverse_tail_head_distribution(triples_np)

    print(f"\n[{split_name}] 逆关系占比")
    print(f"- 三元组总数: {stats['total_triples']}")
    print(
        f"- 至少存在一个逆关系的三元组数: {stats['inverse_matched_triples']} "
        f"({stats['ratio_inverse_triple']:.4%})"
    )
    print(
        f"- 逆关系四元组实例数 (h,r1,t,r2): {stats['inverse_quadruples']} "
        f"(相对三元组数占比 {stats['ratio_inverse_quad']:.4%})"
    )

    top_pairs = stats["relation_pair_counter"].most_common(5)
    if top_pairs:
        print("- 最常见逆关系对 (r1 -> r2) 前5:")
        for i, ((r1, r2), c) in enumerate(top_pairs, 1):
            print(f"  {i}. {relation_labels[r1]} -> {relation_labels[r2]}: {c}")
    else:
        print("- 未发现逆关系对")

    print(f"\n[{split_name}] 在逆关系条件下的 (h,r) 多尾 n 分布")
    print(f"- 满足条件的 unique(h,r) 数量: {stats['total_hr_with_inverse']}")
    print(f"- n 的平均值: {stats['avg_tail_n']:.4f}")
    if stats["tail_n_counter"]:
        print("- n 分布（前10个最常见）:")
        for n, c in stats["tail_n_counter"].most_common(10):
            ratio = c / stats["total_hr_with_inverse"] if stats["total_hr_with_inverse"] else 0.0
            print(f"  n={n}: {c} ({ratio:.2%})")
    else:
        print("- 无满足逆关系条件的 (h,r) 样本")

    print(f"\n[{split_name}] 在逆关系条件下的 (r,t) 多头 n 分布")
    print(f"- 满足条件的 unique(r,t) 数量: {stats['total_rt_with_inverse']}")
    print(f"- n 的平均值: {stats['avg_head_n']:.4f}")
    if stats["head_n_counter"]:
        print("- n 分布（前10个最常见）:")
        for n, c in stats["head_n_counter"].most_common(10):
            ratio = c / stats["total_rt_with_inverse"] if stats["total_rt_with_inverse"] else 0.0
            print(f"  n={n}: {c} ({ratio:.2%})")
    else:
        print("- 无满足逆关系条件的 (r,t) 样本")

KINSHIP: 逆关系占比与多尾/多头分布统计

[train] 逆关系占比
- 三元组总数: 8544
- 至少存在一个逆关系的三元组数: 6778 (79.3305%)
- 逆关系四元组实例数 (h,r1,t,r2): 6778 (相对三元组数占比 79.3305%)
- 最常见逆关系对 (r1 -> r2) 前5:
  1. term7 -> term16: 390
  2. term16 -> term7: 390
  3. term16 -> term8: 350
  4. term8 -> term16: 350
  5. term18 -> term18: 344

[train] 在逆关系条件下的 (h,r) 多尾 n 分布
- 满足条件的 unique(h,r) 数量: 1605
- n 的平均值: 5.2629
- n 分布（前10个最常见）:
  n=2: 228 (14.21%)
  n=1: 218 (13.58%)
  n=3: 198 (12.34%)
  n=4: 156 (9.72%)
  n=6: 155 (9.66%)
  n=5: 149 (9.28%)
  n=7: 126 (7.85%)
  n=8: 86 (5.36%)
  n=9: 79 (4.92%)
  n=10: 57 (3.55%)

[train] 在逆关系条件下的 (r,t) 多头 n 分布
- 满足条件的 unique(r,t) 数量: 1380
- n 的平均值: 6.1377
- n 分布（前10个最常见）:
  n=1: 195 (14.13%)
  n=2: 174 (12.61%)
  n=3: 132 (9.57%)
  n=4: 122 (8.84%)
  n=5: 121 (8.77%)
  n=8: 89 (6.45%)
  n=6: 89 (6.45%)
  n=7: 81 (5.87%)
  n=10: 65 (4.71%)
  n=9: 64 (4.64%)

[valid] 逆关系占比
- 三元组总数: 1068
- 至少存在一个逆关系的三元组数: 82 (7.6779%)
- 逆关系四元组实例数 (h,r1,t,r2): 82 (相对三元组数占比 7.6779%)
- 最常见逆关系对 (r1 -> r2) 前5:
  1. 

In [1]:
from collections import defaultdict
import numpy as np
from pykeen.datasets import Kinships

print("=" * 70)
print("KINSHIP: term 语义可解释性分析")
print("=" * 70)

dataset = Kinships()
all_triples = dataset.merged().mapped_triples.cpu().numpy()
relation_labels = dataset.training.relation_id_to_label

# (h, t, r) 索引，便于检查反向/同向模式
pairs_by_r = defaultdict(set)
for h, r, t in all_triples:
    pairs_by_r[int(r)].add((int(h), int(t)))

print("\n说明：")
print("- PyKEEN 的 Kinships 在数据中仅提供匿名关系名 term0...term25。")
print("- 数据集中不包含官方的 'termX -> mother/father/...' 显式映射表。")
print("- 下方给出可解释线索：每个 term 的规模、对称性，以及最可能的反向关系。")

print("\n[1] 每个 term 的基础统计 + 对称性")
print(f"{'relation':<10} | {'triples':>8} | {'symmetric_ratio':>16}")
print("-" * 44)

for rid in sorted(relation_labels.keys()):
    pairs = pairs_by_r[rid]
    total = len(pairs)
    if total == 0:
        sym_ratio = 0.0
    else:
        sym_cnt = sum((t, h) in pairs for (h, t) in pairs)
        sym_ratio = sym_cnt / total
    print(f"{relation_labels[rid]:<10} | {total:>8} | {sym_ratio:>16.2%}")

print("\n[2] 每个 term 最可能的反向关系候选（基于 pair reverse 重合度）")
print("定义：score(r1->r2) = |{(h,t) in r1 且 (t,h) in r2}| / |r1|")
print(f"{'r1':<10} | {'best_inverse':<12} | {'score':>8}")
print("-" * 40)

rids = sorted(relation_labels.keys())
for r1 in rids:
    p1 = pairs_by_r[r1]
    if not p1:
        print(f"{relation_labels[r1]:<10} | {'N/A':<12} | {0.0:>8.2%}")
        continue

    best_r2 = None
    best_score = -1.0
    for r2 in rids:
        p2 = pairs_by_r[r2]
        overlap = sum((t, h) in p2 for (h, t) in p1)
        score = overlap / len(p1)
        if score > best_score:
            best_score = score
            best_r2 = r2

    print(f"{relation_labels[r1]:<10} | {relation_labels[best_r2]:<12} | {best_score:>8.2%}")

print("\n[3] 示例：每个 term 展示前 3 个 (h,t) 对（entity 为匿名编号）")
for rid in rids:
    pairs = list(pairs_by_r[rid])[:3]
    pair_text = ', '.join([f'({h},{t})' for h, t in pairs]) if pairs else '(none)'
    print(f"- {relation_labels[rid]}: {pair_text}")

print("\n结论：")
print("若你需要准确的人类语义标签（如 mother/father/brother），")
print("需要使用 Kinships 的原始语义说明来源或外部文献映射；仅凭 PyKEEN 当前文件无法一一确定。")

KINSHIP: term 语义可解释性分析

说明：
- PyKEEN 的 Kinships 在数据中仅提供匿名关系名 term0...term25。
- 数据集中不包含官方的 'termX -> mother/father/...' 显式映射表。
- 下方给出可解释线索：每个 term 的规模、对称性，以及最可能的反向关系。

[1] 每个 term 的基础统计 + 对称性
relation   |  triples |  symmetric_ratio
--------------------------------------------
term0      |      228 |           90.35%
term1      |      489 |           48.67%
term10     |      505 |            5.94%
term11     |      739 |            2.98%
term12     |      299 |           28.09%
term13     |      447 |           29.08%
term14     |       43 |            0.00%
term15     |      943 |            0.85%
term16     |     1256 |            4.46%
term17     |      392 |           65.82%
term18     |      569 |           93.85%
term19     |       13 |            0.00%
term2      |      231 |            0.00%
term20     |      272 |           83.82%
term21     |      142 |           45.07%
term22     |      193 |           89.12%
term24     |        2 |            0.00%
term25     |        6 |   

/home/amax/miniconda3/envs/nvembed/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### $\texttt{KINSHIP}$数据筛选

In [8]:
from collections import defaultdict
from pykeen.datasets import Kinships

print("=" * 70)
print("KINSHIP Test 筛选：按‘存在即成立’的对称/可逆规则")
print("=" * 70)

dataset = Kinships()
train_np = dataset.training.mapped_triples.cpu().numpy()
test_np = dataset.testing.mapped_triples.cpu().numpy()
relation_labels = dataset.training.relation_id_to_label

# 1) 构建 train 索引
pairs_by_r = defaultdict(set)  # r -> {(h,t)}
pair_to_relations = defaultdict(set)  # (h,t) -> {r}
for h, r, t in train_np:
    h, r, t = int(h), int(r), int(t)
    pairs_by_r[r].add((h, t))
    pair_to_relations[(h, t)].add(r)

rids = sorted(relation_labels.keys())

# 2) 关系级判定（存在即成立）
symmetric_relations = set()
inverse_pairs = set()  # (r1, r2), r1 != r2

for r in rids:
    # 若存在任意一对 (h,t) 使 (h,r,t) 和 (t,r,h) 同时在 train 中，则判为对称关系
    if any((t, h) in pairs_by_r[r] for (h, t) in pairs_by_r[r]):
        symmetric_relations.add(r)

for r1 in rids:
    for r2 in rids:
        if r1 == r2:
            continue
        # 若存在任意 (h,t) 使 (h,r1,t) 和 (t,r2,h) 同时在 train 中，则判为可逆关系对
        found = any((t, h) in pairs_by_r[r2] for (h, t) in pairs_by_r[r1])
        if found:
            inverse_pairs.add((r1, r2))

# 3) 用 train 证据筛 test 三元组；同时统计 unique(h,r) 问题数
keep_by_symmetry = []
keep_by_inverse = []
keep_union = []

for h, r1, t in test_np:
    h, r1, t = int(h), int(r1), int(t)
    reverse_relations = pair_to_relations.get((t, h), set())

    # 条件A：r1 被判为对称关系，且该样本在 train 中有反向 (t,r1,h)
    cond_sym = (r1 in symmetric_relations) and (r1 in reverse_relations)

    # 条件B：存在 r2，使 r1 与 r2 被判为可逆，且 train 中有 (t,r2,h)
    cond_inv = any((r1, r2) in inverse_pairs and (r2 in reverse_relations) for r2 in rids)

    if cond_sym:
        keep_by_symmetry.append((h, r1, t))
    if cond_inv:
        keep_by_inverse.append((h, r1, t))
    if cond_sym or cond_inv:
        keep_union.append((h, r1, t))

total_test_triples = len(test_np)
total_test_queries = len({(int(h), int(r)) for h, r, t in test_np})

sym_triples = len(keep_by_symmetry)
inv_triples = len(keep_by_inverse)
union_triples = len(keep_union)

sym_queries = len({(h, r) for h, r, t in keep_by_symmetry})
inv_queries = len({(h, r) for h, r, t in keep_by_inverse})
union_queries = len({(h, r) for h, r, t in keep_union})

print("\n[关系判定结果（仅基于 train）]")
print("- 判定规则：存在至少一条反向事实即成立")
print(f"- 判定为对称的关系数: {len(symmetric_relations)}")
print(f"- 判定为可逆关系对数: {len(inverse_pairs)}")

print("\n[筛选后剩余 Test 样本数量]")
print(f"- Test 三元组总数: {total_test_triples}")
print(f"- Test unique(h,r) 问题总数: {total_test_queries}")
print(f"- 仅对称性保留: {sym_triples} triples, {sym_queries} unique(h,r)")
print(f"- 仅可逆性保留: {inv_triples} triples, {inv_queries} unique(h,r)")
print(f"- 对称性∪可逆性保留: {union_triples} triples, {union_queries} unique(h,r)")

if total_test_triples:
    print(f"- 保留比例（按 triples）: {union_triples / total_test_triples:.4%}")
if total_test_queries:
    print(f"- 保留比例（按 unique(h,r)）: {union_queries / total_test_queries:.4%}")

if symmetric_relations:
    print("\n判定为对称的关系:")
    for r in sorted(list(symmetric_relations)):
        print(f"- {relation_labels[r]}")
       
print(len(symmetric_relations))
if inverse_pairs:
    print("\n判定为可逆的关系对（最多10对）:")
    shown = 0
    for r1, r2 in sorted(inverse_pairs):
        print(f"- {relation_labels[r1]} <-> {relation_labels[r2]}")
        shown += 1
        if shown >= 10:
            break

KINSHIP Test 筛选：按‘存在即成立’的对称/可逆规则

[关系判定结果（仅基于 train）]
- 判定规则：存在至少一条反向事实即成立
- 判定为对称的关系数: 20
- 判定为可逆关系对数: 210

[筛选后剩余 Test 样本数量]
- Test 三元组总数: 1074
- Test unique(h,r) 问题总数: 744
- 仅对称性保留: 254 triples, 208 unique(h,r)
- 仅可逆性保留: 616 triples, 462 unique(h,r)
- 对称性∪可逆性保留: 870 triples, 648 unique(h,r)
- 保留比例（按 triples）: 81.0056%
- 保留比例（按 unique(h,r)）: 87.0968%

判定为对称的关系:
- term0
- term1
- term10
- term11
- term12
- term13
- term15
- term16
- term17
- term18
- term20
- term21
- term22
- term3
- term4
- term5
- term6
- term7
- term8
- term9
20

判定为可逆的关系对（最多10对）:
- term0 <-> term1
- term0 <-> term10
- term0 <-> term11
- term0 <-> term15
- term0 <-> term16
- term0 <-> term2
- term0 <-> term6
- term1 <-> term0
- term1 <-> term10
- term1 <-> term11


In [9]:
from collections import defaultdict
from pykeen.datasets import Kinships

print("=" * 70)
print("KINSHIP 对称关系审计：存在性 vs 强对称")
print("=" * 70)

dataset = Kinships()
train_np = dataset.training.mapped_triples.cpu().numpy()
relation_labels = dataset.training.relation_id_to_label

pairs_by_r = defaultdict(set)
for h, r, t in train_np:
    pairs_by_r[int(r)].add((int(h), int(t)))

print(f"{'relation':<8} | {'total_pairs':>10} | {'sym_pairs':>9} | {'sym_ratio':>9}")
print("-" * 50)

rows = []
for rid in sorted(relation_labels.keys()):
    pairs = pairs_by_r[rid]
    total = len(pairs)
    sym_pairs = sum((t, h) in pairs for (h, t) in pairs)
    sym_ratio = (sym_pairs / total) if total else 0.0
    rows.append((rid, total, sym_pairs, sym_ratio))
    print(f"{relation_labels[rid]:<8} | {total:>10} | {sym_pairs:>9} | {sym_ratio:>9.2%}")

exist_symmetric = [relation_labels[rid] for rid, _, sym_pairs, _ in rows if sym_pairs > 0]
strong_symmetric_50 = [relation_labels[rid] for rid, _, _, ratio in rows if ratio >= 0.50]
strong_symmetric_80 = [relation_labels[rid] for rid, _, _, ratio in rows if ratio >= 0.80]

print("\n[汇总]")
print(f"- 按‘存在至少1条对称对’判定: {len(exist_symmetric)} 个关系")
print(f"- 按‘对称比例 >= 50%’判定: {len(strong_symmetric_50)} 个关系")
print(f"- 按‘对称比例 >= 80%’判定: {len(strong_symmetric_80)} 个关系")

print("\n对称比例 >= 80% 的关系:")
if strong_symmetric_80:
    for name in strong_symmetric_80:
        print(f"- {name}")
else:
    print("- 无")

print("\n说明：")
print("- 你之前看到的 20 个，是‘存在性判定’，只要出现过至少一条就算。")
print("- 这个标准会偏宽松，容易把弱对称关系也算进去。")

KINSHIP 对称关系审计：存在性 vs 强对称
relation | total_pairs | sym_pairs | sym_ratio
--------------------------------------------------
term0    |        185 |       134 |    72.43%
term1    |        384 |       140 |    36.46%
term10   |        392 |        16 |     4.08%
term11   |        600 |        14 |     2.33%
term12   |        236 |        46 |    19.49%
term13   |        367 |        80 |    21.80%
term14   |         34 |         0 |     0.00%
term15   |        757 |         6 |     0.79%
term16   |       1004 |        38 |     3.78%
term17   |        320 |       160 |    50.00%
term18   |        460 |       344 |    74.78%
term19   |         10 |         0 |     0.00%
term2    |        183 |         0 |     0.00%
term20   |        209 |       128 |    61.24%
term21   |        106 |        42 |    39.62%
term22   |        153 |       104 |    67.97%
term24   |          2 |         0 |     0.00%
term25   |          6 |         0 |     0.00%
term3    |        299 |       142 |    47.49%
te

### 解释性诊断：为什么 Inverse Relation 不总是单答案？

In [1]:
from collections import defaultdict
from pykeen.datasets import Kinships

print("=" * 80)
print("KINSHIP: 逆关系唯一性与多答案诊断")
print("=" * 80)

dataset = Kinships()
train_np = dataset.training.mapped_triples.cpu().numpy()
valid_np = dataset.validation.mapped_triples.cpu().numpy()
test_np = dataset.testing.mapped_triples.cpu().numpy()
all_np = dataset.merged().mapped_triples.cpu().numpy()

relation_labels = dataset.training.relation_id_to_label
rids = sorted(relation_labels.keys())

# 关系到pair索引
pairs_train = defaultdict(set)
pairs_all = defaultdict(set)
for h, r, t in train_np:
    pairs_train[int(r)].add((int(h), int(t)))
for h, r, t in all_np:
    pairs_all[int(r)].add((int(h), int(t)))

# -----------------------------
# A) 一个关系是否可能有多个逆关系？
# -----------------------------
exist_inverse_map = defaultdict(list)   # 存在即成立
strict_inverse_map = defaultdict(list)  # 全覆盖成立

for r1 in rids:
    p1 = pairs_train[r1]
    if not p1:
        continue
    for r2 in rids:
        if r1 == r2:
            continue
        p2 = pairs_train[r2]
        if not p2:
            continue

        overlap = sum((t, h) in p2 for (h, t) in p1)
        if overlap > 0:
            exist_inverse_map[r1].append((r2, overlap / len(p1), overlap))
        if overlap == len(p1):
            strict_inverse_map[r1].append((r2, 1.0, overlap))

multi_exist = {r1: lst for r1, lst in exist_inverse_map.items() if len(lst) >= 2}
multi_strict = {r1: lst for r1, lst in strict_inverse_map.items() if len(lst) >= 2}

print("\n[A] 一个关系是否有多个逆关系")
print(f"- 按存在即成立(overlap>0): 有候选逆关系的r数量 = {len(exist_inverse_map)}")
print(f"- 其中拥有 >=2 个逆候选的r数量 = {len(multi_exist)}")
print(f"- 按严格全覆盖(overlap=100%): 有逆关系的r数量 = {len(strict_inverse_map)}")
print(f"- 其中拥有 >=2 个严格逆关系的r数量 = {len(multi_strict)}")

if multi_exist:
    print("\n存在即成立下，示例（最多5个关系）:")
    shown = 0
    for r1, lst in multi_exist.items():
        lst_sorted = sorted(lst, key=lambda x: x[1], reverse=True)
        top_text = ", ".join([f"{relation_labels[r2]}(cover={cov:.2%}, overlap={ov})" for r2, cov, ov in lst_sorted[:4]])
        print(f"- {relation_labels[r1]} -> {top_text}")
        shown += 1
        if shown >= 5:
            break

# -----------------------------
# B) 为什么 inverse 查询不是单答案？
#    核心看 (r, t) -> {h} 的多头性
# -----------------------------
def rt_head_stats(triples_np):
    rt_to_h = defaultdict(set)
    for h, r, t in triples_np:
        rt_to_h[(int(r), int(t))].add(int(h))

    n_values = [len(v) for v in rt_to_h.values()]
    total_rt = len(n_values)
    single = sum(n == 1 for n in n_values)
    multi = sum(n >= 2 for n in n_values)
    avg_n = (sum(n_values) / total_rt) if total_rt else 0.0
    return total_rt, single, multi, avg_n

train_total_rt, train_single, train_multi, train_avg = rt_head_stats(train_np)
all_total_rt, all_single, all_multi, all_avg = rt_head_stats(all_np)

print("\n[B] (r,t)->head 多答案统计")
print(f"- Train: unique(r,t)={train_total_rt}, 单头={train_single} ({train_single/train_total_rt:.2%}), 多头={train_multi} ({train_multi/train_total_rt:.2%}), 平均n={train_avg:.4f}")
print(f"- All  : unique(r,t)={all_total_rt}, 单头={all_single} ({all_single/all_total_rt:.2%}), 多头={all_multi} ({all_multi/all_total_rt:.2%}), 平均n={all_avg:.4f}")

# -----------------------------
# C) 复现实验：train里 n=1，但 test 真值不同 的比例
# -----------------------------
# 先取“最强逆关系映射”：对每个r1，选覆盖率最高的r2
best_inv = {}
for r1, lst in exist_inverse_map.items():
    r2_best, cov_best, ov_best = sorted(lst, key=lambda x: x[1], reverse=True)[0]
    best_inv[r1] = (r2_best, cov_best)

heads_by_train_rt = defaultdict(set)
for h, r, t in train_np:
    heads_by_train_rt[(int(r), int(t))].add(int(h))

n1_queries = 0
n1_train_eq_test = 0
examples_mismatch = []

for h, r1, t in test_np:
    h = int(h)
    r1 = int(r1)
    t = int(t)

    if r1 not in best_inv:
        continue
    r2, cov = best_inv[r1]

    # inverse query: (t, r2, ?)
    train_heads = heads_by_train_rt.get((r2, h), set())
    if len(train_heads) != 1:
        continue

    n1_queries += 1
    only_head = next(iter(train_heads))
    if only_head == t:
        n1_train_eq_test += 1
    elif len(examples_mismatch) < 8:
        examples_mismatch.append((h, r1, t, r2, only_head, cov))

print("\n[C] train中 inverse 查询为n=1 时，与test真值的一致性")
print(f"- n=1 的 test 查询数量: {n1_queries}")
if n1_queries > 0:
    ratio = n1_train_eq_test / n1_queries
    print(f"- train唯一head == test真值: {n1_train_eq_test}/{n1_queries} = {ratio:.2%}")

if examples_mismatch:
    print("\n不一致示例（最多8条）:")
    for i, (h, r1, t, r2, only_head, cov) in enumerate(examples_mismatch, start=1):
        print(
            f"{i}. 原test: ({h}, {relation_labels[r1]}, {t}) | 逆关系取 {relation_labels[r2]} (cover={cov:.2%}) | "
            f"train唯一head={only_head}"
        )

print("\n结论提示:")
print("1) 亲属关系通常不是一一映射，(r,t) 常对应多个 h，因此 inverse 查询天然可能多答案。")
print("2) 在 KINSHIP 中，用 train 学到的‘唯一答案’不保证与 test 标注一致。")
print("3) 若用存在即成立定义逆关系，一个关系可能有多个逆候选；需用覆盖率或阈值收紧。")

KINSHIP: 逆关系唯一性与多答案诊断

[A] 一个关系是否有多个逆关系
- 按存在即成立(overlap>0): 有候选逆关系的r数量 = 25
- 其中拥有 >=2 个逆候选的r数量 = 23
- 按严格全覆盖(overlap=100%): 有逆关系的r数量 = 1
- 其中拥有 >=2 个严格逆关系的r数量 = 0

存在即成立下，示例（最多5个关系）:
- term0 -> term1(cover=3.78%, overlap=7), term10(cover=1.08%, overlap=2), term15(cover=1.08%, overlap=2), term16(cover=1.08%, overlap=2)
- term1 -> term2(cover=33.33%, overlap=128), term0(cover=1.82%, overlap=7), term11(cover=1.82%, overlap=7), term9(cover=0.78%, overlap=3)
- term10 -> term11(cover=61.48%, overlap=241), term9(cover=12.76%, overlap=50), term2(cover=0.77%, overlap=3), term0(cover=0.51%, overlap=2)
- term11 -> term10(cover=40.17%, overlap=241), term9(cover=33.50%, overlap=201), term1(cover=1.17%, overlap=7), term8(cover=0.50%, overlap=3)
- term12 -> term13(cover=22.46%, overlap=53), term17(cover=9.32%, overlap=22), term7(cover=9.32%, overlap=22), term14(cover=6.36%, overlap=15)

[B] (r,t)->head 多答案统计
- Train: unique(r,t)=1442, 单头=247 (17.13%), 多头=1195 (82.87%), 平均n=5.9251
- All  : unique(r,

/home/amax/miniconda3/envs/nvembed/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


$P_r=\{(h,t)\mid (h,r,t)\in \mathcal{D}_{train}\}$

$\text{overlap}(r_1,r_2)=\left|{(h,t)\in P_{r_1}\mid (t,h)\in P_{r_2}}\right|$

$\text{cover}(r_1\to r_2)=\frac{\text{overlap}(r_1,r_2)}{|P_{r_1}|}$

# $\texttt{Countries}$

In [3]:
from pykeen.datasets import Countries

# 检查 Countries 数据集中是否存在 neighborOf(c1, c2) 和 neighborOf(c2, c1) 同时出现的情况
dataset = Countries()
triples = dataset.merged().label_triples(dataset.training.mapped_triples)

neighbor_pairs = set()
for h, r, t in triples:
    if "neighbor" in str(r).lower():
        neighbor_pairs.add((str(h), str(t)))

reciprocal_pairs = sorted(
    {
        (h, t)
        for h, t in neighbor_pairs
        if (t, h) in neighbor_pairs and h <= t
    }
)

print(f"neighborOf 三元组总数: {len(neighbor_pairs)}")
print(f"存在双向成对的 neighborOf 数量: {len(reciprocal_pairs)}")

if reciprocal_pairs:
    print("示例:")
    for h, t in reciprocal_pairs[:20]:
        print(f"neighborOf({h}, {t}) 和 neighborOf({t}, {h})")
else:
    print("没有发现任何头尾互换后同时存在的 neighborOf 对。")

neighborOf 三元组总数: 648
存在双向成对的 neighborOf 数量: 320
示例:
neighborOf(afghanistan, china) 和 neighborOf(china, afghanistan)
neighborOf(afghanistan, iran) 和 neighborOf(iran, afghanistan)
neighborOf(afghanistan, pakistan) 和 neighborOf(pakistan, afghanistan)
neighborOf(afghanistan, tajikistan) 和 neighborOf(tajikistan, afghanistan)
neighborOf(afghanistan, turkmenistan) 和 neighborOf(turkmenistan, afghanistan)
neighborOf(afghanistan, uzbekistan) 和 neighborOf(uzbekistan, afghanistan)
neighborOf(albania, greece) 和 neighborOf(greece, albania)
neighborOf(albania, kosovo) 和 neighborOf(kosovo, albania)
neighborOf(albania, macedonia) 和 neighborOf(macedonia, albania)
neighborOf(albania, montenegro) 和 neighborOf(montenegro, albania)
neighborOf(algeria, libya) 和 neighborOf(libya, algeria)
neighborOf(algeria, mali) 和 neighborOf(mali, algeria)
neighborOf(algeria, mauritania) 和 neighborOf(mauritania, algeria)
neighborOf(algeria, morocco) 和 neighborOf(morocco, algeria)
neighborOf(algeria, niger) 和 neighborOf(nig

In [4]:
import numpy as np

# ==========================================
# 1. 定义矩阵 K (2行关系 x 5列实体)
# ==========================================
# 行: 0=属于(R1), 1=邻居(R2)
# 列: 0=中国(A), 1=韩国(B), 2=日本(C), 3=亚洲(D), 4=地球(E)
K = np.zeros((2, 5))
K[0, 0] = 1  # 仅 (R1, 中国) 激活 (对应1-index的{1,1})

print("=== 矩阵 K (初始激活状态) ===")
print(K)
print()

# ==========================================
# 2. 将 K 展平为向量 (长度 2*5=10)
# ==========================================
# 展平顺序: [R1中国, R1韩国, R1日本, R1亚洲, R1地球, R2中国, ..., R2地球]
k_vec = K.flatten()

print("=== K 展平后的向量 ===")
print(k_vec)
print()

# ==========================================
# 3. 定义权重矩阵 W (10 x 10)
# ==========================================
W = np.zeros((10, 10))

# 规则 1: 中国 -(属于)-> 亚洲
# 即: 输入位置0 (R1中国) 激活 输出位置3 (R1亚洲)
W[3, 0] = 1

# 规则 2: 中国 -(属于)-> 地球 (或者你可以定义 亚洲->地球 进行传递)
# 即: 输入位置0 (R1中国) 激活 输出位置4 (R1地球)
W[4, 0] = 1

# ==========================================
# 4. 计算 W @ K (矩阵乘法)
# ==========================================
result_vec = W @ k_vec

# 将结果 reshape 回 2x5 矩阵
result_matrix = result_vec.reshape(2, 5)

print("=== W @ K 的结果矩阵 ===")
print(result_matrix)
print()

# ==========================================
# 5. 输出激活位置坐标
# ==========================================
# 找出非零位置 (coords 是 0-index)
coords = np.argwhere(result_matrix != 0)

print("=== 被激活的位置 (1-index) ===")
for (row, col) in coords:
    print(f"(行 {row+1}, 列 {col+1})")

=== 矩阵 K (初始激活状态) ===
[[1. 0. 0. 0. 0.]
 [0. 0. 0. 0. 0.]]

=== K 展平后的向量 ===
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]

=== W @ K 的结果矩阵 ===
[[0. 0. 0. 1. 1.]
 [0. 0. 0. 0. 0.]]

=== 被激活的位置 (1-index) ===
(行 1, 列 4)
(行 1, 列 5)


## Other KGE Models On \texttt{KINSHIP,COUNTRIES,KINSHIP11990\_EXTENDED}
- 主要是保证配置与在Nations上的配置一致

In [1]:
import time
import torch
import numpy as np
import pandas as pd
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
from pykeen.evaluation import RankBasedEvaluator
from pykeen.models import TransE, DistMult, ComplEx, RotatE, ConvE, RESCAL
from pykeen.datasets import Kinships

device_kinship = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 100)
print("KINSHIP: 其他 KGE 模型测试 (使用带有早期停止和针对性超参调优验证)")
print("=" * 100)
print(f"Device: {device_kinship}")

# Load KINSHIP dataset
kinship_dataset = Kinships()
entity_to_id = kinship_dataset.training.entity_to_id
relation_to_id = kinship_dataset.training.relation_to_id
train_triples = kinship_dataset.training.mapped_triples.cpu().numpy()
valid_triples = kinship_dataset.validation.mapped_triples.cpu().numpy()
test_triples = kinship_dataset.testing.mapped_triples.cpu().numpy()

def build_tf_kinship(triples, create_inverse=False):
    return TriplesFactory(
        mapped_triples=np.asarray(triples, dtype=np.int64),
        entity_to_id=entity_to_id,
        relation_to_id=relation_to_id,
        create_inverse_triples=create_inverse,
    )

def metric_dict(metric_results):
    return {
        "MR": float(metric_results.get_metric("mean_rank")),
        "MRR": float(metric_results.get_metric("mean_reciprocal_rank")),
        "Hits@1": float(metric_results.get_metric("hits_at_1")) * 100,
        "Hits@3": float(metric_results.get_metric("hits_at_3")) * 100,
        "Hits@10": float(metric_results.get_metric("hits_at_10")) * 100,
    }

# 用于最后 evaluation 的 base (没有 Inverse Triples)
train_eval_tf = build_tf_kinship(train_triples, create_inverse=False)
valid_eval_tf = build_tf_kinship(valid_triples, create_inverse=False)
test_eval_tf = build_tf_kinship(test_triples, create_inverse=False)

def evaluate_split_kinship(model, split_tf, filtered=False):
    evaluator = RankBasedEvaluator(filtered=filtered)
    kwargs = {"batch_size": 128}
    if filtered:
        kwargs["additional_filter_triples"] = [
            train_eval_tf.mapped_triples,
            valid_eval_tf.mapped_triples,
            test_eval_tf.mapped_triples,
        ]
    metric_results = evaluator.evaluate(
        model=model,
        mapped_triples=split_tf.mapped_triples,
        **kwargs,
    )
    return metric_dict(metric_results)

# 包含了代表性模型 + 对其维度和学习率做了初步调优的配置
model_specs_kinship = [
    {
        "name": "TransE",
        "model": "TransE",
        "training_loop": "slcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100, "scoring_fct_norm": 1},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "DistMult",
        "model": "DistMult",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "ComplEx",
        "model": "ComplEx",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "RotatE",
        "model": "RotatE",
        "training_loop": "slcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 5e-4},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "ConvE",
        "model": "ConvE",
        "training_loop": "lcwa",
        "create_inverse": True,  # ConvE 强制要求开启 inverse triples
        "model_kwargs": {
            "embedding_dim": 100,
            "output_channels": 32,
            "input_dropout": 0.2,
            "feature_map_dropout": 0.2,
            "output_dropout": 0.3,
        },
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "RESCAL",
        "model": "RESCAL",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
]

kge_kinship_results = {}
seed_kinship = 42

for spec in model_specs_kinship:
    print("\n" + "=" * 90)
    print(f"训练模型: {spec['name']}")
    print("=" * 90)

    train_tf = build_tf_kinship(train_triples, create_inverse=spec["create_inverse"])
    valid_tf = build_tf_kinship(valid_triples, create_inverse=False)

    start_time = time.time()
    
    # 借助 pykeen 的 pipeline 整合自动带有 Early Stopping
    try:
        result = pipeline(
            training=train_tf,
            validation=valid_tf,
            testing=test_eval_tf,
            model=spec["model"],
            model_kwargs=spec["model_kwargs"],
            training_loop=spec["training_loop"],
            optimizer="adam",
            optimizer_kwargs=spec["optimizer_kwargs"],
            training_kwargs=spec["train_kwargs"],
            stopper="early",
            stopper_kwargs={
                "frequency": 10,
                "patience": 10,  # KINSHIP数据集较小可以稍微等久一点容忍度
                "relative_delta": 0.002,
                "metric": "mean_reciprocal_rank",
            },
            evaluator="RankBasedEvaluator",
            evaluator_kwargs={"filtered": True},
            random_seed=seed_kinship,
            device=device_kinship,
        )
        elapsed = time.time() - start_time

        model = result.model
        filtered_metrics = evaluate_split_kinship(model, test_eval_tf, filtered=True)

        # === 测试 PyKEEN 模型的纯推理速度 ===
        hr_batch = torch.tensor([[h, r] for h, r, t in test_triples], device=device_kinship)
        model.eval()
        with torch.no_grad():
            # warm-up
            _ = model.score_t(hr_batch[:1])
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            start_infer = time.time()
            _ = model.score_t(hr_batch)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            pure_infer_time = time.time() - start_infer
        
        kge_kinship_results[spec["name"]] = {
            "training_time": elapsed,
            "filtered": filtered_metrics,
            "pure_infer_time": pure_infer_time,
        }
        
        print(f"{spec['name']} | filtered MRR={filtered_metrics['MRR']:.4f} | train_time={elapsed:.1f}s | infer_time={pure_infer_time:.4f}s")
        
        # 清理显存避免爆内存
        del model
        del result
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f"训练 {spec['name']} 失败: {str(e)}")

print("\n" + "=" * 100)
print("KINSHIP: KGE 模型与 NSR 结果汇总对比")
print("=" * 100)

rows = []

# 把我们前面的 Enhanced NSR 结果放进来
if "enhanced_filtered" in globals():
    rows.append({
        "Model": "⭐ Enhanced NSR",
        "MR": enhanced_filtered.get("MR", 0.0),
        "MRR": enhanced_filtered.get("MRR", 0.0),
        "Hits@1": enhanced_filtered.get("Hits@1", 0.0),
        "Hits@3": enhanced_filtered.get("Hits@3", 0.0),
        "Hits@10": enhanced_filtered.get("Hits@10", 0.0),
        "Train Time(s)": f"{enhanced_filt_t:.2f}" if "enhanced_filt_t" in globals() else "-",
        "Infer Time(s)": f"{pure_infer_time:.4f}" if "pure_infer_time" in globals() else "-",
    })

for model_name, results in kge_kinship_results.items():
    rows.append({
        "Model": model_name,
        "MR": results["filtered"]["MR"],
        "MRR": results["filtered"]["MRR"],
        "Hits@1": results["filtered"]["Hits@1"],
        "Hits@3": results["filtered"]["Hits@3"],
        "Hits@10": results["filtered"]["Hits@10"],
        "Train Time(s)": f"{results['training_time']:.1f}",
        "Infer Time(s)": f"{results.get('pure_infer_time', 0):.4f}",
    })

df_rows = pd.DataFrame(rows).sort_values("MRR", ascending=False).reset_index(drop=True)
print("\nFiltered Test (Sorted by MRR)")
print("-" * 110)
print(df_rows.to_string(index=False, formatters={
    "MR": lambda x: f"{x:.4f}",
    "MRR": lambda x: f"{x:.4f}",
    "Hits@1": lambda x: f"{x:.2f}%",
    "Hits@3": lambda x: f"{x:.2f}%",
    "Hits@10": lambda x: f"{x:.2f}%",
    "Infer Time(s)": lambda x: f"{x}",
}))

print("\n" + "=" * 100)
print("KINSHIP 对比测试完成")
print("=" * 100)


/home/amax/miniconda3/envs/nvembed/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KINSHIP: 其他 KGE 模型测试 (使用带有早期停止和针对性超参调优验证)
Device: cuda

训练模型: TransE


Training epochs on cuda:0:   6%|▌         | 9/150 [00:05<01:09,  2.02epoch/s, loss=0.508, prev_loss=0.542]INFO:pykeen.evaluation.evaluator:Evaluation took 0.06s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.10182078182697296. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-08d92201-201e-4230-80a5-03248977578a.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  13%|█▎        | 19/150 [00:09<00:59,  2.22epoch/s, loss=0.318, prev_loss=0.326]INFO:pykeen.evaluation.evaluator:Evaluation took 0.03s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 20: 0.16906501352787018. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-08d92201-201e-4230-80a5-03248977578a.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 20.
Training epochs on cuda:0:  19%|█▉        | 29/150 [00:14<00:55,  2.17epoch/s, 

TransE | filtered MRR=0.2557 | train_time=65.7s | infer_time=0.0002s

训练模型: DistMult


Training epochs on cuda:0:   6%|▌         | 9/150 [00:02<00:38,  3.62epoch/s, loss=0.695, prev_loss=0.75] INFO:pykeen.evaluation.evaluator:Evaluation took 0.03s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.3108212947845459. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-d249a10e-b881-4f08-90ef-d9ae0975c372.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  33%|███▎      | 49/150 [00:11<00:24,  4.05epoch/s, loss=0.286, prev_loss=0.288]INFO:pykeen.evaluation.evaluator:Evaluation took 0.03s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 50: 0.35735848546028137. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-d249a10e-b881-4f08-90ef-d9ae0975c372.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 50.
Training epochs on cuda:0:  39%|███▉      | 59/150 [00:14<00:22,  4.07epoch/s, l

...

RotatE | filtered MRR=0.7547 | train_time=74.5s | infer_time=0.0002s

训练模型: ConvE


Training epochs on cuda:0:   0%|          | 0/150 [00:00<?, ?epoch/s]INFO:pykeen.triples.triples_factory:Creating inverse triples.
INFO:pykeen.training.training_loop:Dropping last (incomplete) batch each epoch (1/97 (1.03%) batches).
Training epochs on cuda:0:   6%|▌         | 9/150 [00:03<00:49,  2.87epoch/s, loss=0.395, prev_loss=0.465]INFO:pykeen.evaluation.evaluator:Evaluation took 0.03s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.3005639612674713. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-9637ada4-ee25-46ea-92d0-aeb137c22ead.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  13%|█▎        | 19/150 [00:07<00:45,  2.85epoch/s, loss=0.134, prev_loss=0.14] INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 20: 0.5458153486251831. Saved model weights to /home/amax/.data/pykeen/check

ConvE | filtered MRR=0.7859 | train_time=66.7s | infer_time=0.0004s

训练模型: RESCAL


Training epochs on cuda:0:   6%|▌         | 9/150 [00:02<00:38,  3.68epoch/s, loss=17.8, prev_loss=17.8]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.07100504636764526. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-d78d456a-8608-4967-a097-f7faa0f8db31.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  13%|█▎        | 19/150 [00:05<00:30,  4.27epoch/s, loss=16.9, prev_loss=17]  INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 20: 0.07343974709510803. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-d78d456a-8608-4967-a097-f7faa0f8db31.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 20.
Training epochs on cuda:0:  19%|█▉        | 29/150 [00:07<00:28,  4.19epoch/s, loss

RESCAL | filtered MRR=0.3358 | train_time=38.4s | infer_time=0.0002s

KINSHIP: KGE 模型与 NSR 结果汇总对比

Filtered Test (Sorted by MRR)
--------------------------------------------------------------------------------------------------------------
   Model      MR    MRR Hits@1 Hits@3 Hits@10 Train Time(s) Infer Time(s)
   ConvE  2.2467 0.7859 67.41% 87.76%  98.04%          66.7        0.0004
  RotatE  2.3431 0.7547 62.57% 86.13%  97.53%          74.5        0.0002
DistMult  6.1378 0.4002 23.00% 45.20%  82.87%          34.3        0.0001
 ComplEx 10.3557 0.3421 18.58% 38.69%  71.37%          36.9        0.0002
  RESCAL  6.7793 0.3358 15.50% 38.31%  78.82%          38.4        0.0002
  TransE  8.7253 0.2557  0.84% 38.22%  76.16%          65.7        0.0002

KINSHIP 对比测试完成


In [2]:
import time
import torch
import numpy as np
import pandas as pd
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
from pykeen.evaluation import RankBasedEvaluator
from pykeen.models import TransE, DistMult, ComplEx, RotatE, ConvE, RESCAL
from pykeen.datasets import Countries

device_country = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 100)
print("COUNTRIES: KGE 模型测试 (使用早期停止和针对性超参调优验证)")
print("=" * 100)
print(f"Device: {device_country}")

# Load COUNTRIES dataset
country_dataset = Countries()
entity_to_id = country_dataset.training.entity_to_id
relation_to_id = country_dataset.training.relation_to_id
train_triples = country_dataset.training.mapped_triples.cpu().numpy()
valid_triples = country_dataset.validation.mapped_triples.cpu().numpy()
test_triples = country_dataset.testing.mapped_triples.cpu().numpy()

def build_tf_country(triples, create_inverse=False):
    return TriplesFactory(
        mapped_triples=np.asarray(triples, dtype=np.int64),
        entity_to_id=entity_to_id,
        relation_to_id=relation_to_id,
        create_inverse_triples=create_inverse,
    )

def metric_dict(metric_results):
    return {
        "MR": float(metric_results.get_metric("mean_rank")),
        "MRR": float(metric_results.get_metric("mean_reciprocal_rank")),
        "Hits@1": float(metric_results.get_metric("hits_at_1")) * 100,
        "Hits@3": float(metric_results.get_metric("hits_at_3")) * 100,
        "Hits@10": float(metric_results.get_metric("hits_at_10")) * 100,
    }

# 用于最后 evaluation 的 base (没有 Inverse Triples)
train_eval_tf = build_tf_country(train_triples, create_inverse=False)
valid_eval_tf = build_tf_country(valid_triples, create_inverse=False)
test_eval_tf = build_tf_country(test_triples, create_inverse=False)

def evaluate_split_country(model, split_tf, filtered=False):
    evaluator = RankBasedEvaluator(filtered=filtered)
    kwargs = {"batch_size": 128}
    if filtered:
        kwargs["additional_filter_triples"] = [
            train_eval_tf.mapped_triples,
            valid_eval_tf.mapped_triples,
            test_eval_tf.mapped_triples,
        ]
    metric_results = evaluator.evaluate(
        model=model,
        mapped_triples=split_tf.mapped_triples,
        **kwargs,
    )
    return metric_dict(metric_results)

# 与 KINSHIP 完全相同的模型配置
model_specs_country = [
    {
        "name": "TransE",
        "model": "TransE",
        "training_loop": "slcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100, "scoring_fct_norm": 1},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "DistMult",
        "model": "DistMult",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "ComplEx",
        "model": "ComplEx",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "RotatE",
        "model": "RotatE",
        "training_loop": "slcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 5e-4},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "ConvE",
        "model": "ConvE",
        "training_loop": "lcwa",
        "create_inverse": True,
        "model_kwargs": {
            "embedding_dim": 100,
            "output_channels": 32,
            "input_dropout": 0.2,
            "feature_map_dropout": 0.2,
            "output_dropout": 0.3,
        },
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "RESCAL",
        "model": "RESCAL",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
]

kge_country_results = {}
seed_country = 42

for spec in model_specs_country:
    print("\n" + "=" * 90)
    print(f"训练模型: {spec['name']}")
    print("=" * 90)

    train_tf = build_tf_country(train_triples, create_inverse=spec["create_inverse"])
    valid_tf = build_tf_country(valid_triples, create_inverse=False)

    start_time = time.time()
    
    try:
        result = pipeline(
            training=train_tf,
            validation=valid_tf,
            testing=test_eval_tf,
            model=spec["model"],
            model_kwargs=spec["model_kwargs"],
            training_loop=spec["training_loop"],
            optimizer="adam",
            optimizer_kwargs=spec["optimizer_kwargs"],
            training_kwargs=spec["train_kwargs"],
            stopper="early",
            stopper_kwargs={
                "frequency": 10,
                "patience": 10,
                "relative_delta": 0.002,
                "metric": "mean_reciprocal_rank",
            },
            evaluator="RankBasedEvaluator",
            evaluator_kwargs={"filtered": True},
            random_seed=seed_country,
            device=device_country,
        )
        elapsed = time.time() - start_time

        model = result.model
        filtered_metrics = evaluate_split_country(model, test_eval_tf, filtered=True)

        # 纯推理速度测试
        hr_batch = torch.tensor([[h, r] for h, r, t in test_triples], device=device_country)
        model.eval()
        with torch.no_grad():
            _ = model.score_t(hr_batch[:1])
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            start_infer = time.time()
            _ = model.score_t(hr_batch)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            pure_infer_time = time.time() - start_infer
        
        kge_country_results[spec["name"]] = {
            "training_time": elapsed,
            "filtered": filtered_metrics,
            "pure_infer_time": pure_infer_time,
        }
        
        print(f"{spec['name']} | filtered MRR={filtered_metrics['MRR']:.4f} | train_time={elapsed:.1f}s | infer_time={pure_infer_time:.4f}s")
        
        del model
        del result
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f"训练 {spec['name']} 失败: {str(e)}")

print("\n" + "=" * 100)
print("COUNTRIES: KGE 模型结果汇总")
print("=" * 100)

rows = []

for model_name, results in kge_country_results.items():
    rows.append({
        "Model": model_name,
        "MR": results["filtered"]["MR"],
        "MRR": results["filtered"]["MRR"],
        "Hits@1": results["filtered"]["Hits@1"],
        "Hits@3": results["filtered"]["Hits@3"],
        "Hits@10": results["filtered"]["Hits@10"],
        "Train Time(s)": f"{results['training_time']:.1f}",
        "Infer Time(s)": f"{results.get('pure_infer_time', 0):.4f}",
    })

df_rows = pd.DataFrame(rows).sort_values("MRR", ascending=False).reset_index(drop=True)
print("\nFiltered Test (Sorted by MRR)")
print("-" * 110)
print(df_rows.to_string(index=False, formatters={
    "MR": lambda x: f"{x:.4f}",
    "MRR": lambda x: f"{x:.4f}",
    "Hits@1": lambda x: f"{x:.2f}%",
    "Hits@3": lambda x: f"{x:.2f}%",
    "Hits@10": lambda x: f"{x:.2f}%",
    "Infer Time(s)": lambda x: f"{x}",
}))

print("\n" + "=" * 100)
print("COUNTRIES 测试完成")
print("=" * 100)

INFO:pykeen.pipeline.api:Using device: cuda
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.stoppers.early_stopping:Inferred checkpoint path for best model weights: /home/amax/.data/pykeen/checkpoints/best-model-weights-9e9844c9-dad7-4e48-aa7c-ef01a6cb624b.pt


COUNTRIES: KGE 模型测试 (使用早期停止和针对性超参调优验证)
Device: cuda

训练模型: TransE


Training epochs on cuda:0:   6%|▌         | 9/150 [00:01<00:21,  6.70epoch/s, loss=0.267, prev_loss=0.244]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.12860798835754395. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-9e9844c9-dad7-4e48-aa7c-ef01a6cb624b.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  13%|█▎        | 19/150 [00:03<00:19,  6.74epoch/s, loss=0.104, prev_loss=0.0864]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 20: 0.15180452167987823. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-9e9844c9-dad7-4e48-aa7c-ef01a6cb624b.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 20.
Training epochs on cuda:0:  19%|█▉        | 29/150 [00:04<00:18,  6.68epoch/s,

TransE | filtered MRR=0.3355 | train_time=22.5s | infer_time=0.0006s

训练模型: DistMult


Training epochs on cuda:0:   6%|▌         | 9/150 [00:01<00:19,  7.24epoch/s, loss=0.976, prev_loss=0.982]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.031984154134988785. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-536daeee-7034-4f0c-9b45-b8b7f5a4cfa5.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  13%|█▎        | 19/150 [00:02<00:17,  7.47epoch/s, loss=0.857, prev_loss=0.874]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 20: 0.46351954340934753. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-536daeee-7034-4f0c-9b45-b8b7f5a4cfa5.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 20.
Training epochs on cuda:0:  19%|█▉        | 29/150 [00:04<00:16,  7.52epoch/s,

...

RotatE | filtered MRR=0.6727 | train_time=23.1s | infer_time=0.0001s

训练模型: ConvE


Training epochs on cuda:0:   0%|          | 0/150 [00:00<?, ?epoch/s]INFO:pykeen.triples.triples_factory:Creating inverse triples.
INFO:pykeen.training.training_loop:Dropping last (incomplete) batch each epoch (1/19 (5.26%) batches).
Training epochs on cuda:0:   6%|▌         | 9/150 [00:01<00:19,  7.06epoch/s, loss=0.192, prev_loss=0.216]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.35754039883613586. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-631d3bed-fa72-4721-a9e2-673a3768142b.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  13%|█▎        | 19/150 [00:02<00:18,  7.06epoch/s, loss=0.0665, prev_loss=0.0768]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 20: 0.36013856530189514. Saved model weights to /home/amax/.data/pykeen/c

ConvE | filtered MRR=0.6586 | train_time=21.7s | infer_time=0.0003s

训练模型: RESCAL


Training epochs on cuda:0:   6%|▌         | 9/150 [00:01<00:17,  7.88epoch/s, loss=17.8, prev_loss=18]  INFO:pykeen.evaluation.evaluator:Evaluation took 0.01s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 10: 0.18605268001556396. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-a4bc2e7f-f358-4a7f-aeac-b77a15550c39.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 10.
Training epochs on cuda:0:  26%|██▌       | 39/150 [00:05<00:15,  7.15epoch/s, loss=16.3, prev_loss=16.3]INFO:pykeen.evaluation.evaluator:Evaluation took 0.02s seconds
INFO:pykeen.stoppers.early_stopping:New best result at epoch 40: 0.20451878011226654. Saved model weights to /home/amax/.data/pykeen/checkpoints/best-model-weights-a4bc2e7f-f358-4a7f-aeac-b77a15550c39.pt
INFO:pykeen.training.training_loop:=> Saved checkpoint after having finished epoch 40.
Training epochs on cuda:0:  39%|███▉      | 59/150 [00:08<00:13,  6.94epoch/s, loss

RESCAL | filtered MRR=0.1958 | train_time=20.9s | infer_time=0.0001s

COUNTRIES: KGE 模型结果汇总

Filtered Test (Sorted by MRR)
--------------------------------------------------------------------------------------------------------------
   Model       MR    MRR Hits@1 Hits@3 Hits@10 Train Time(s) Infer Time(s)
DistMult   2.8958 0.8404 75.00% 91.67%  97.92%          20.7        0.0001
  RotatE   2.4167 0.6727 45.83% 87.50%  97.92%          23.1        0.0001
   ConvE   3.6667 0.6586 47.92% 79.17%  93.75%          21.7        0.0003
  TransE  11.7708 0.3355  0.00% 58.33%  85.42%          22.5        0.0006
  RESCAL  35.7292 0.1958  6.25% 14.58%  52.08%          20.9        0.0001
 ComplEx 114.0833 0.0202  0.00%  0.00%   4.17%          22.1        0.0002

COUNTRIES 测试完成


In [ ]:
import time
import torch
import numpy as np
import pandas as pd
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
from pykeen.evaluation import RankBasedEvaluator
from pykeen.models import TransE, DistMult, ComplEx, RotatE, ConvE, RESCAL

device_kinship90 = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("=" * 100)
print("KINSHIP_1990: 其他 KGE 模型测试 (使用带有早期停止和针对性超参调优验证)")
print("=" * 100)
print(f"Device: {device_kinship90}")

# =============================
# 1. 加载并解析 KINSHIP_1990 本地数据
# =============================
import re
import os
from collections import defaultdict

DATA_DIR = "KINSHIP_1990"

def parse_kinship_file(filepath):
    triples = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            # 格式: relation(head, tail)
            m = re.match(r"([^(]+)\(([^,]+),\s*([^)]+)\)", line)
            if m:
                r, h, t = m.group(1).strip(), m.group(2).strip(), m.group(3).strip()
                triples.append((h, r, t))
    return triples

train_raw = parse_kinship_file(os.path.join(DATA_DIR, "train.data"))
valid_raw = parse_kinship_file(os.path.join(DATA_DIR, "valid.data"))
test_raw  = parse_kinship_file(os.path.join(DATA_DIR, "test.data"))

# 构建全局 entity / relation 映射（以 train 为主，补充 valid/test 中出现的新实体/关系）
all_triples_raw = train_raw + valid_raw + test_raw
entities = sorted({e for h, r, t in all_triples_raw for e in (h, t)})
relations = sorted({r for h, r, t in all_triples_raw})

entity_to_id = {e: i for i, e in enumerate(entities)}
relation_to_id = {r: i for i, r in enumerate(relations)}

def map_triples(triples_raw):
    return np.array(
        [[entity_to_id[h], relation_to_id[r], entity_to_id[t]] for h, r, t in triples_raw],
        dtype=np.int64,
    )

train_triples = map_triples(train_raw)
valid_triples = map_triples(valid_raw)
test_triples  = map_triples(test_raw)

print(f"加载完成 -> entities: {len(entity_to_id)}, relations: {len(relation_to_id)}")
print(f"           train: {len(train_triples)}, valid: {len(valid_triples)}, test: {len(test_triples)}")

# =============================
# 2. 构建 TriplesFactory 与评估函数
# =============================
def build_tf_kinship90(triples, create_inverse=False):
    return TriplesFactory(
        mapped_triples=np.asarray(triples, dtype=np.int64),
        entity_to_id=entity_to_id,
        relation_to_id=relation_to_id,
        create_inverse_triples=create_inverse,
    )

def metric_dict(metric_results):
    return {
        "MR": float(metric_results.get_metric("mean_rank")),
        "MRR": float(metric_results.get_metric("mean_reciprocal_rank")),
        "Hits@1": float(metric_results.get_metric("hits_at_1")) * 100,
        "Hits@3": float(metric_results.get_metric("hits_at_3")) * 100,
        "Hits@10": float(metric_results.get_metric("hits_at_10")) * 100,
    }

# 用于最后 evaluation 的 base (没有 Inverse Triples)
train_eval_tf = build_tf_kinship90(train_triples, create_inverse=False)
valid_eval_tf = build_tf_kinship90(valid_triples, create_inverse=False)
test_eval_tf  = build_tf_kinship90(test_triples,  create_inverse=False)

def evaluate_split_kinship90(model, split_tf, filtered=False):
    evaluator = RankBasedEvaluator(filtered=filtered)
    kwargs = {"batch_size": 128}
    if filtered:
        kwargs["additional_filter_triples"] = [
            train_eval_tf.mapped_triples,
            valid_eval_tf.mapped_triples,
            test_eval_tf.mapped_triples,
        ]
    metric_results = evaluator.evaluate(
        model=model,
        mapped_triples=split_tf.mapped_triples,
        **kwargs,
    )
    return metric_dict(metric_results)

# =============================
# 3. 模型配置
# =============================
model_specs_kinship90 = [
    {
        "name": "TransE",
        "model": "TransE",
        "training_loop": "slcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100, "scoring_fct_norm": 1},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "DistMult",
        "model": "DistMult",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "ComplEx",
        "model": "ComplEx",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "RotatE",
        "model": "RotatE",
        "training_loop": "slcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 5e-4},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "ConvE",
        "model": "ConvE",
        "training_loop": "lcwa",
        "create_inverse": True,  # ConvE 强制要求开启 inverse triples
        "model_kwargs": {
            "embedding_dim": 100,
            "output_channels": 32,
            "input_dropout": 0.2,
            "feature_map_dropout": 0.2,
            "output_dropout": 0.3,
        },
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
    {
        "name": "RESCAL",
        "model": "RESCAL",
        "training_loop": "lcwa",
        "create_inverse": False,
        "model_kwargs": {"embedding_dim": 100},
        "optimizer_kwargs": {"lr": 1e-3},
        "train_kwargs": {"num_epochs": 150, "batch_size": 32},
    },
]

kge_kinship90_results = {}
seed_kinship90 = 42

for spec in model_specs_kinship90:
    print("\n" + "=" * 90)
    print(f"训练模型: {spec['name']}")
    print("=" * 90)

    train_tf = build_tf_kinship90(train_triples, create_inverse=spec["create_inverse"])
    valid_tf = build_tf_kinship90(valid_triples, create_inverse=False)

    start_time = time.time()
    
    # 借助 pykeen 的 pipeline 整合自动带有 Early Stopping
    try:
        result = pipeline(
            training=train_tf,
            validation=valid_tf,
            testing=test_eval_tf,
            model=spec["model"],
            model_kwargs=spec["model_kwargs"],
            training_loop=spec["training_loop"],
            optimizer="adam",
            optimizer_kwargs=spec["optimizer_kwargs"],
            training_kwargs=spec["train_kwargs"],
            stopper="early",
            stopper_kwargs={
                "frequency": 10,
                "patience": 10,
                "relative_delta": 0.002,
                "metric": "mean_reciprocal_rank",
            },
            evaluator="RankBasedEvaluator",
            evaluator_kwargs={"filtered": True},
            random_seed=seed_kinship90,
            device=device_kinship90,
        )
        elapsed = time.time() - start_time

        model = result.model
        filtered_metrics = evaluate_split_kinship90(model, test_eval_tf, filtered=True)

        # === 测试 PyKEEN 模型的纯推理速度 ===
        hr_batch = torch.tensor([[h, r] for h, r, t in test_triples], device=device_kinship90)
        model.eval()
        with torch.no_grad():
            # warm-up
            _ = model.score_t(hr_batch[:1])
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            start_infer = time.time()
            _ = model.score_t(hr_batch)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            pure_infer_time = time.time() - start_infer
        
        kge_kinship90_results[spec["name"]] = {
            "training_time": elapsed,
            "filtered": filtered_metrics,
            "pure_infer_time": pure_infer_time,
        }
        
        print(f"{spec['name']} | filtered MRR={filtered_metrics['MRR']:.4f} | train_time={elapsed:.1f}s | infer_time={pure_infer_time:.4f}s")
        
        # 清理显存避免爆内存
        del model
        del result
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
    except Exception as e:
        print(f"训练 {spec['name']} 失败: {str(e)}")

print("\n" + "=" * 100)
print("KINSHIP_1990: KGE 模型与 NSR 结果汇总对比")
print("=" * 100)

rows = []

# 把我们前面的 Enhanced NSR 结果放进来
if "enhanced_filtered" in globals():
    rows.append({
        "Model": "⭐ Enhanced NSR",
        "MR": enhanced_filtered.get("MR", 0.0),
        "MRR": enhanced_filtered.get("MRR", 0.0),
        "Hits@1": enhanced_filtered.get("Hits@1", 0.0),
        "Hits@3": enhanced_filtered.get("Hits@3", 0.0),
        "Hits@10": enhanced_filtered.get("Hits@10", 0.0),
        "Train Time(s)": f"{enhanced_filt_t:.2f}" if "enhanced_filt_t" in globals() else "-",
        "Infer Time(s)": f"{pure_infer_time:.4f}" if "pure_infer_time" in globals() else "-",
    })

for model_name, results in kge_kinship90_results.items():
    rows.append({
        "Model": model_name,
        "MR": results["filtered"]["MR"],
        "MRR": results["filtered"]["MRR"],
        "Hits@1": results["filtered"]["Hits@1"],
        "Hits@3": results["filtered"]["Hits@3"],
        "Hits@10": results["filtered"]["Hits@10"],
        "Train Time(s)": f"{results['training_time']:.1f}",
        "Infer Time(s)": f"{results.get('pure_infer_time', 0):.4f}",
    })

df_rows = pd.DataFrame(rows).sort_values("MRR", ascending=False).reset_index(drop=True)
print("\nFiltered Test (Sorted by MRR)")
print("-" * 110)
print(df_rows.to_string(index=False, formatters={
    "MR": lambda x: f"{x:.4f}",
    "MRR": lambda x: f"{x:.4f}",
    "Hits@1": lambda x: f"{x:.2f}%",
    "Hits@3": lambda x: f"{x:.2f}%",
    "Hits@10": lambda x: f"{x:.2f}%",
    "Infer Time(s)": lambda x: f"{x}",
}))

print("\n" + "=" * 100)
print("KINSHIP_1990 对比测试完成")
print("=" * 100)
